# Timestomping Detection Tool - Feature Engineering

This notebook extracts forensic features from the merged dataset for timestomping detection.

## Input

- `data_merged.csv` from notebook 01

Feature Categories (matching Phase 2):
1. Forensic Patterns (16 features - Oh et al.)
2. Cross-Artifact Validation (4 features - Oh et al.)
3. Temporal Features (3 features - Oh et al.)
4. File Characteristics (9 features - Oh et al.)
5. Timestamp Features (2 features - Oh et al.)
6. System Pattern Recognition (2 features - Data-Driven)
7. Timestamp Parsing (9 features - old Random Forest)
8. Source Classification (3 features - old Random Forest)

Total: 48 features

## Output

- `data_features.csv` - Dataset with 31 extracted features ready for detection


## Cell 1: Imports and Setup

In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully
Pandas version: 2.3.2
NumPy version: 2.3.3


## Cell 2: User Configuration

**EDIT THIS SECTION** if you changed paths in notebook 01

In [125]:
# Cell 2: User Configuration

# Input file (output from notebook 01)
INPUT_DIR = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/11-PE'
INPUT_FILE = "data_merged.csv"

# Output directory
OUTPUT_DIR = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/11-PE/revised 02'
OUTPUT_FILE = "data_features.csv"

# Construct paths
input_path = os.path.join(INPUT_DIR, INPUT_FILE)
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

print("Configuration loaded")
print("-" * 80)
print(f"Input file: {input_path}")
print(f"  Exists: {os.path.exists(input_path)}")
print(f"Output file: {output_path}")
print("-" * 80)

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Merged data not found: {input_path}")


Configuration loaded
--------------------------------------------------------------------------------
Input file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/11-PE/data_merged.csv
  Exists: True
Output file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/11-PE/revised 02/data_features.csv
--------------------------------------------------------------------------------


## Cell 3: Load Merged Dataset

In [126]:
# Cell 3: Load Merged Dataset

print("=" * 80)
print("LOADING MERGED DATASET")
print("=" * 80)

# Load merged data
df = pd.read_csv(input_path, low_memory=False)

print(f"\nDataset loaded successfully")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

# Display column names
print(f"\nAvailable columns:")
print(df.columns.tolist())


LOADING MERGED DATASET

Dataset loaded successfully
  Records: 894
  Columns: 40
  Memory usage: 1.10 MB

Available columns:
['merge_key', 'lf_lsn', 'EventTime(UTC+8)', 'lf_event', 'lf_detail', 'File/Directory Name', 'lf_full_path', 'lf_creation_time', 'lf_modified_time', 'lf_mft_modified_time', 'lf_accessed_time', 'Redo', 'Target VCN', 'Cluster Index', 'usn_event_time', 'usn_usn', 'usn_filename', 'usn_full_path', 'usn_event_info', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'usn_file_reference_number', 'usn_parent_file_reference_number', 'suspicious_category', 'suspicious_detail', 'suspicious_source', 'lf_event_count', 'usn_event_count', 'has_logfile_evidence', 'has_usnjrnl_evidence', 'cross_artifact_validation_score', 'has_suspicious_label', 'filename', 'full_path', 'zero_in_nanoseconds_lf', 'zero_in_nanoseconds_suspicious', 'zero_in_nanoseconds_combined', 'is_flagged_suspicious', 'ground_truth_label']


## Cell 4: Extract Forensic Pattern Features (15 features)

Based on Oh et al. timestomping indicators


In [127]:
# Cell 4: Extract Forensic Pattern Features (CORRECTED - Matches Phase 2)

print("\n" + "=" * 80)
print("EXTRACTING FORENSIC PATTERN FEATURES")
print("=" * 80)

# Feature 1: Zero nanoseconds in LogFile (from lf_detail field)
print("\n1. Zero in nanoseconds (LogFile)")
df['zero_in_nanoseconds_lf'] = df['lf_detail'].fillna('').str.contains(
    'Zero in 100-nanoseconds', case=False, na=False
).astype(int)
print(f"   Detected: {df['zero_in_nanoseconds_lf'].sum()} files")

# Feature 2: Zero nanoseconds in Suspicious (from suspicious_detail field)
print("\n2. Zero in nanoseconds (Suspicious)")
df['zero_in_nanoseconds_suspicious'] = df['suspicious_detail'].fillna('').str.contains(
    'Zero in 100-nanoseconds', case=False, na=False
).astype(int)
print(f"   Detected: {df['zero_in_nanoseconds_suspicious'].sum()} files")

# Feature 3: Zero nanoseconds combined (LogFile OR Suspicious)
print("\n3. Zero in nanoseconds (Combined)")
df['zero_in_nanoseconds'] = (
    (df['zero_in_nanoseconds_lf'] == 1) | 
    (df['zero_in_nanoseconds_suspicious'] == 1)
).astype(int)
print(f"   Detected: {df['zero_in_nanoseconds'].sum()} files")

# Feature 4: Time reversal event
print("\n4. Time reversal event")
df['time_reversal_event'] = df['lf_event'].fillna('').str.contains(
    'Time Reversal', case=False, na=False
).astype(int)
print(f"   Detected: {df['time_reversal_event'].sum()} files")

# Feature 5: Basic info changed (UsnJrnl)
print("\n5. Basic info changed")
df['basic_info_changed'] = df['usn_event_info'].fillna('').str.contains(
    'Basic_Info_Change', case=False, na=False
).astype(int)
print(f"   Detected: {df['basic_info_changed'].sum()} files")

# Feature 6: Using another file's timestamp
print("\n6. Using another file's timestamp")
df['using_another_timestamp'] = df['lf_detail'].fillna('').str.contains(
    'Using another', case=False, na=False
).astype(int)
print(f"   Detected: {df['using_another_timestamp'].sum()} files")

# Feature 7: SI timestamp changed
print("\n7. SI timestamp changed")
df['si_timestamp_changed'] = df['lf_detail'].fillna('').str.contains(
    r'\$SI timestamp', case=False, na=False, regex=True
).astype(int)
print(f"   Detected: {df['si_timestamp_changed'].sum()} files")

# Feature 8: Update resident value
print("\n8. Update resident value")
df['update_resident_value'] = df['lf_event'].fillna('').str.contains(
    'Update Resident Value', case=False, na=False
).astype(int)
print(f"   Detected: {df['update_resident_value'].sum()} files")

# Features 9-12: Timestamp modification patterns (CORRECTED NAMES)
print("\n9-12. Timestamp modification patterns")
df['creation_time_modified'] = df['lf_detail'].fillna('').str.contains(
    'CreationTime', case=False, na=False
).astype(int)
print(f"   creation_time_modified: {df['creation_time_modified'].sum()} files")

df['modified_time_modified'] = df['lf_detail'].fillna('').str.contains(
    'ModifiedTime', case=False, na=False
).astype(int)
print(f"   modified_time_modified: {df['modified_time_modified'].sum()} files")

df['accessed_time_modified'] = df['lf_detail'].fillna('').str.contains(
    'AccessedTime', case=False, na=False
).astype(int)
print(f"   accessed_time_modified: {df['accessed_time_modified'].sum()} files")

df['mft_time_modified'] = df['lf_detail'].fillna('').str.contains(
    'MFTModified', case=False, na=False
).astype(int)
print(f"   mft_time_modified: {df['mft_time_modified'].sum()} files")

# Feature 13: Timestamp changed to past
print("\n13. Timestamp changed to past")
df['timestamp_changed_to_past'] = df['lf_detail'].fillna('').str.contains(
    r'->', case=False, na=False, regex=True
).astype(int)
print(f"   Detected: {df['timestamp_changed_to_past'].sum()} files")

# Feature 14: Multiple timestamps changed
print("\n14. Multiple timestamps changed")
df['multiple_timestamps_changed'] = (
    df['creation_time_modified'] + 
    df['modified_time_modified'] + 
    df['accessed_time_modified'] + 
    df['mft_time_modified']
)
print(f"   Average timestamps modified: {df['multiple_timestamps_changed'].mean():.2f}")

# Feature 15: Same as another file
print("\n15. Same as another file")
df['same_as_another_file'] = df['lf_detail'].fillna('').str.contains(
    'same as', case=False, na=False
).astype(int)
print(f"   Detected: {df['same_as_another_file'].sum()} files")

# Feature 16: Refined detection - Zero nano + time reversal
print("\n16. Zero nano + time reversal (combined)")
df['zero_nano_time_reversal'] = (
    (df['zero_in_nanoseconds'] == 1) & 
    (df['time_reversal_event'] == 1)
).astype(int)
print(f"   Detected: {df['zero_nano_time_reversal'].sum()} files")

print("\nForensic pattern features extracted: 16 features")



EXTRACTING FORENSIC PATTERN FEATURES

1. Zero in nanoseconds (LogFile)
   Detected: 0 files

2. Zero in nanoseconds (Suspicious)
   Detected: 0 files

3. Zero in nanoseconds (Combined)
   Detected: 0 files

4. Time reversal event
   Detected: 78 files

5. Basic info changed
   Detected: 894 files

6. Using another file's timestamp
   Detected: 0 files

7. SI timestamp changed
   Detected: 0 files

8. Update resident value
   Detected: 0 files

9-12. Timestamp modification patterns
   creation_time_modified: 1 files
   modified_time_modified: 78 files
   accessed_time_modified: 1 files
   mft_time_modified: 78 files

13. Timestamp changed to past
   Detected: 78 files

14. Multiple timestamps changed
   Average timestamps modified: 0.18

15. Same as another file
   Detected: 1 files

16. Zero nano + time reversal (combined)
   Detected: 0 files

Forensic pattern features extracted: 16 features


## Cell 5: Extract Cross-Artifact Validation Features (3 features)

Validate detections across LogFile and UsnJrnl


In [128]:
# Cell 5: Extract Cross-Artifact Validation Features (FIXED - Production Mode)

print("\n" + "=" * 80)
print("EXTRACTING CROSS-ARTIFACT VALIDATION FEATURES")
print("=" * 80)

# Feature 1: Has LogFile evidence
print("\n1. Has LogFile evidence")
df['has_logfile_evidence'] = df['lf_event'].notna().astype(int)
print(f"   Files with LogFile evidence: {df['has_logfile_evidence'].sum()}")

# Feature 2: Has UsnJrnl evidence
print("\n2. Has UsnJrnl evidence")
df['has_usnjrnl_evidence'] = df['usn_event_info'].notna().astype(int)
print(f"   Files with UsnJrnl evidence: {df['has_usnjrnl_evidence'].sum()}")

# Feature 3: Cross-artifact validation score
print("\n3. Cross-artifact validation score")
def calculate_cross_artifact_score(row):
    """
    Calculate cross-artifact validation score based on Oh et al. methodology.
    
    Score = 1.0: Both LogFile AND UsnJrnl detected activity (HIGH confidence)
    Score = 0.5: Only ONE source detected activity (MEDIUM confidence)
    Score = 0.0: No evidence (BENIGN or insufficient data)
    """
    score = 0.0
    if row['has_logfile_evidence'] == 1:
        score += 0.5
    if row['has_usnjrnl_evidence'] == 1:
        score += 0.5
    return score

df['cross_artifact_validation_score'] = df.apply(calculate_cross_artifact_score, axis=1)

print(f"   Score distribution:")
print(df['cross_artifact_validation_score'].value_counts().sort_index())

# Feature 4: Cross-artifact detected (both sources have evidence)
print("\n4. Cross-artifact detected")
# Note: In production, we don't have suspicious labels
# So we use presence of evidence from both sources as indicator
df['cross_artifact_detected'] = (
    (df['has_logfile_evidence'] == 1) & 
    (df['has_usnjrnl_evidence'] == 1)
).astype(int)
print(f"   Files detected by both sources: {df['cross_artifact_detected'].sum()}")

print("\nCross-artifact validation features extracted: 4 features")



EXTRACTING CROSS-ARTIFACT VALIDATION FEATURES

1. Has LogFile evidence
   Files with LogFile evidence: 78

2. Has UsnJrnl evidence
   Files with UsnJrnl evidence: 894

3. Cross-artifact validation score
   Score distribution:
cross_artifact_validation_score
0.5    816
1.0     78
Name: count, dtype: int64

4. Cross-artifact detected
   Files detected by both sources: 78

Cross-artifact validation features extracted: 4 features


## Cell 6: Extract File Characteristic Features (9 features)

File type, path, and location indicators


In [129]:
# Cell 6: Extract File Characteristic Features (CORRECTED - Matches Phase 2)

print("\n" + "=" * 80)
print("EXTRACTING FILE CHARACTERISTIC FEATURES")
print("=" * 80)

# Feature 1: Is executable
print("\n1. Is executable")
df['is_executable'] = df['filename'].fillna('').str.lower().str.endswith(
    ('.exe', '.dll', '.sys', '.scr', '.bat', '.cmd', '.ps1')
).astype(int)
print(f"   Executables: {df['is_executable'].sum()} files")

# Feature 2: Is document
print("\n2. Is document")
df['is_document'] = df['filename'].fillna('').str.lower().str.endswith(
    ('.docx', '.doc', '.pdf', '.txt', '.rtf', '.xlsx', '.xls', '.pptx', '.ppt')
).astype(int)
print(f"   Documents: {df['is_document'].sum()} files")

# Feature 3: Is archive
print("\n3. Is archive")
df['is_archive'] = df['filename'].fillna('').str.lower().str.endswith(
    ('.zip', '.rar', '.7z', '.tar', '.gz', '.bz2')
).astype(int)
print(f"   Archives: {df['is_archive'].sum()} files")

# Feature 4: Is image
print("\n4. Is image")
df['is_image'] = df['filename'].fillna('').str.lower().str.endswith(
    ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.ico', '.svg')
).astype(int)
print(f"   Images: {df['is_image'].sum()} files")

# Feature 5: Path depth
print("\n5. Path depth")
df['path_depth'] = df['full_path'].fillna('').str.count(r'\\')
print(f"   Average path depth: {df['path_depth'].mean():.2f}")

# Feature 6: Filename length
print("\n6. Filename length")
df['filename_length'] = df['filename'].fillna('').str.len()
print(f"   Average filename length: {df['filename_length'].mean():.2f}")

# Feature 7: In temp directory
print("\n7. In temp directory")
df['in_temp_directory'] = df['full_path'].fillna('').str.contains(
    r'\\Temp\\', case=False, na=False, regex=True
).astype(int)
print(f"   Files in temp: {df['in_temp_directory'].sum()}")

# Feature 8: In system directory
print("\n8. In system directory")
df['in_system_directory'] = df['full_path'].fillna('').str.contains(
    r'\\Windows\\', case=False, na=False, regex=True
).astype(int)
print(f"   Files in Windows: {df['in_system_directory'].sum()}")

# Feature 9: In program files
print("\n9. In program files")
df['in_program_files'] = df['full_path'].fillna('').str.contains(
    r'\\Program Files', case=False, na=False, regex=True
).astype(int)
print(f"   Files in Program Files: {df['in_program_files'].sum()}")

print("\nFile characteristic features extracted: 9 features")



EXTRACTING FILE CHARACTERISTIC FEATURES

1. Is executable
   Executables: 99 files

2. Is document
   Documents: 19 files

3. Is archive
   Archives: 3 files

4. Is image
   Images: 7 files

5. Path depth
   Average path depth: 5.34

6. Filename length
   Average filename length: 31.10

7. In temp directory
   Files in temp: 164

8. In system directory
   Files in Windows: 276

9. In program files
   Files in Program Files: 97

File characteristic features extracted: 9 features


## Cell 7: Extract Timestamp Relationship Features (2 features)

Analyze timestamp patterns and relationships


In [130]:
# Cell 7: Extract Timestamp Features (CORRECTED - Matches Phase 2)

print("\n" + "=" * 80)
print("EXTRACTING TIMESTAMP FEATURES")
print("=" * 80)

# Feature 1: Has timestamp data
print("\n1. Has timestamp data")
df['has_timestamp_data'] = (
    df['EventTime(UTC+8)'].notna() | 
    df['usn_event_time'].notna()
).astype(int)
print(f"   Files with timestamp data: {df['has_timestamp_data'].sum()}")

# Feature 2: Timestamp source
print("\n2. Timestamp source")
def get_timestamp_source(row):
    """
    Determine which artifact(s) provided timestamp information.
    Returns: 0=None, 1=LogFile only, 2=UsnJrnl only, 3=Both
    """
    has_lf_time = pd.notna(row.get('EventTime(UTC+8)'))
    has_usn_time = pd.notna(row.get('usn_event_time'))
    
    if has_lf_time and has_usn_time:
        return 3
    elif has_lf_time:
        return 1
    elif has_usn_time:
        return 2
    else:
        return 0

df['timestamp_source'] = df.apply(get_timestamp_source, axis=1)

print(f"   Timestamp source distribution:")
print(f"     None (0): {(df['timestamp_source'] == 0).sum()}")
print(f"     LogFile only (1): {(df['timestamp_source'] == 1).sum()}")
print(f"     UsnJrnl only (2): {(df['timestamp_source'] == 2).sum()}")
print(f"     Both (3): {(df['timestamp_source'] == 3).sum()}")

print("\nTimestamp features extracted: 2 features")



EXTRACTING TIMESTAMP FEATURES

1. Has timestamp data
   Files with timestamp data: 894

2. Timestamp source
   Timestamp source distribution:
     None (0): 0
     LogFile only (1): 0
     UsnJrnl only (2): 816
     Both (3): 78

Timestamp features extracted: 2 features


## Cell 8: Feature Summary and Validation


In [131]:
# Cell 8: Feature Summary and Validation (UPDATED)

print("\n" + "=" * 80)
print("FEATURE EXTRACTION SUMMARY")
print("=" * 80)

# Define all engineered features (MUST match Phase 2 exactly)
ENGINEERED_FEATURES = [
    # Forensic Patterns (16 features)
    'zero_in_nanoseconds_lf',
    'zero_in_nanoseconds_suspicious',
    'zero_in_nanoseconds',
    'time_reversal_event',
    'basic_info_changed',
    'using_another_timestamp',
    'si_timestamp_changed',
    'update_resident_value',
    'creation_time_modified',
    'modified_time_modified',
    'accessed_time_modified',
    'mft_time_modified',
    'timestamp_changed_to_past',
    'multiple_timestamps_changed',
    'same_as_another_file',
    'zero_nano_time_reversal',
    
    # Cross-Artifact Validation (4 features)
    'cross_artifact_detected',
    'has_logfile_evidence',
    'has_usnjrnl_evidence',
    'cross_artifact_validation_score',
    
    # File Characteristics (9 features)
    'is_executable',
    'is_document',
    'is_archive',
    'is_image',
    'path_depth',
    'filename_length',
    'in_temp_directory',
    'in_system_directory',
    'in_program_files',
    
    # Timestamp Features (2 features)
    'has_timestamp_data',
    'timestamp_source',
]

print(f"\nTotal features extracted: {len(ENGINEERED_FEATURES)}")
print(f"Total columns in dataset: {len(df.columns)}")

print("\nFeature categories:")
print("  Forensic Patterns: 16")
print("  Cross-Artifact Validation: 4")
print("  File Characteristics: 9")
print("  Timestamp Features: 2")
print("  " + "-" * 40)
print("  Total: 31 features")

# Verify all features exist
missing_features = [f for f in ENGINEERED_FEATURES if f not in df.columns]
if missing_features:
    print(f"\n  ERROR: Missing features:")
    for feat in missing_features:
        print(f"    - {feat}")
else:
    print("\n  ✅ All 31 features present")

# Check for missing values
missing_values = df[ENGINEERED_FEATURES].isna().sum()
if missing_values.sum() == 0:
    print("  ✅ No missing values in features")
else:
    print(f"  WARNING: Missing values detected:")
    print(missing_values[missing_values > 0])

# Display top features by detection count
print("\nTop 10 features by detection count:")
feature_counts = {}
for col in ENGINEERED_FEATURES:
    if df[col].dtype in ['bool', 'int64']:
        feature_counts[col] = df[col].sum()
    elif col == 'cross_artifact_validation_score':
        feature_counts[col] = (df[col] > 0).sum()

top_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for feat, count in top_features:
    pct = (count / len(df)) * 100 if len(df) > 0 else 0
    print(f"  {feat:40s}: {count:6,} ({pct:5.2f}%)")

print("\n✅ FEATURE EXTRACTION MATCHES PHASE 2 EXACTLY")



FEATURE EXTRACTION SUMMARY

Total features extracted: 31
Total columns in dataset: 66

Feature categories:
  Forensic Patterns: 16
  Cross-Artifact Validation: 4
  File Characteristics: 9
  Timestamp Features: 2
  ----------------------------------------
  Total: 31 features

  ✅ All 31 features present
  ✅ No missing values in features

Top 10 features by detection count:
  filename_length                         : 27,803 (3109.96%)
  path_depth                              :  4,778 (534.45%)
  timestamp_source                        :  1,866 (208.72%)
  basic_info_changed                      :    894 (100.00%)
  has_usnjrnl_evidence                    :    894 (100.00%)
  cross_artifact_validation_score         :    894 (100.00%)
  has_timestamp_data                      :    894 (100.00%)
  in_system_directory                     :    276 (30.87%)
  in_temp_directory                       :    164 (18.34%)
  multiple_timestamps_changed             :    158 (17.67%)

✅ FEATURE EXTR

## Cell 9: Save Feature Dataset


In [132]:
# Cell 9: Save Feature Dataset

print("\n" + "=" * 80)
print("SAVING FEATURE DATASET")
print("=" * 80)

# Save to CSV
df.to_csv(output_path, index=False)

# Verify save
file_size_mb = os.path.getsize(output_path) / 1024 / 1024

print(f"\nDataset saved successfully")
print(f"  Output file: {output_path}")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Features: 31")
print(f"  File size: {file_size_mb:.2f} MB")

print("\n" + "=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)
print("\nNext step: Open '03 - Run Detection.ipynb' to run LightGBM model predictions")



SAVING FEATURE DATASET

Dataset saved successfully
  Output file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/11-PE/revised 02/data_features.csv
  Records: 894
  Columns: 66
  Features: 31
  File size: 0.45 MB

FEATURE ENGINEERING COMPLETE

Next step: Open '03 - Run Detection.ipynb' to run LightGBM model predictions


In [135]:
import pandas as pd

# Load Prototype Tool output for 11-PE
prototype_df = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/Training Datasets/11-PE/revised 02/data_features.csv')

print(f"Prototype Tool features: {len(prototype_df.columns)}")
print(f"Feature columns: {prototype_df.columns.tolist()}")


Prototype Tool features: 66
Feature columns: ['merge_key', 'lf_lsn', 'EventTime(UTC+8)', 'lf_event', 'lf_detail', 'File/Directory Name', 'lf_full_path', 'lf_creation_time', 'lf_modified_time', 'lf_mft_modified_time', 'lf_accessed_time', 'Redo', 'Target VCN', 'Cluster Index', 'usn_event_time', 'usn_usn', 'usn_filename', 'usn_full_path', 'usn_event_info', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'usn_file_reference_number', 'usn_parent_file_reference_number', 'suspicious_category', 'suspicious_detail', 'suspicious_source', 'lf_event_count', 'usn_event_count', 'has_logfile_evidence', 'has_usnjrnl_evidence', 'cross_artifact_validation_score', 'has_suspicious_label', 'filename', 'full_path', 'zero_in_nanoseconds_lf', 'zero_in_nanoseconds_suspicious', 'zero_in_nanoseconds_combined', 'is_flagged_suspicious', 'ground_truth_label', 'zero_in_nanoseconds', 'time_reversal_event', 'basic_info_changed', 'using_another_timestamp', 'si_timestamp_changed', 'update_resident_value', 'creation_time_